<a href="https://colab.research.google.com/github/Bubukisapisa/ML_education/blob/main/HW_2_4_kNN_%D0%9A%D1%80%D0%BE%D1%81%D0%B2%D0%B0%D0%BB%D1%96%D0%B4%D0%B0%D1%86%D1%96%D1%8F_%D1%96_%D1%82%D1%8E%D0%BD%D0%B8%D0%BD%D0%B3_%D0%B3%D1%96%D0%BF%D0%B5%D1%80%D0%BF%D0%B0%D1%80%D0%B0%D0%BC%D0%B5%D1%82%D1%80%D1%96%D0%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням kNearestNeighboors, знайдемо оптимальні гіперпараметри для цього методу і зробимо базові ансамблі. Це дасть змогу порівняти перформанс моделі з попередніми вивченими методами.

0. Зчитайте дані `train.csv` та зробіть препроцесинг використовуючи написаний Вами скрипт `process_bank_churn.py` так, аби в результаті отримати дані в розбитті X_train, train_targets, X_val, val_targets для експериментів.

  Якщо Вам не вдалось реалізувати в завданні `2.3. Дерева прийняття рішень` скрипт `process_bank_churn.py` - можна скористатись готовим скриптом з запропонованого рішення того завдання.

In [2]:
import numpy as np
import pandas as pd
import process_bank_churn as pp
from sklearn.neighbors import KNeighborsClassifier
from google.colab import drive
drive.mount('/content/drive')
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier

Mounted at /content/drive


In [3]:
prep_df = pd.read_csv('drive/MyDrive/ML for people/train.csv')
data = pp.preprocess_data(prep_df)
data['train_x'].head()

/content/process_bank_churn.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_new['Gender'] = df_new['Gender'].replace({'Male': 1, 'Female': 0})


,id,CustomerId,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
7180,7180,15652218.0,0.599045,1.0,0.214286,0.1,0.626382,1.0,1.0,1.0,0.719772,1.0,0.0,0.0
10393,10393,15592937.0,0.603819,0.0,0.375000,0.2,0.848836,1.0,1.0,0.0,0.727603,1.0,0.0,0.0
80,80,15774586.0,0.653938,1.0,0.303571,0.6,0.554522,2.0,1.0,0.0,0.872180,0.0,1.0,0.0
3365,3365,15780572.0,0.568019,1.0,0.714286,0.0,0.000000,2.0,0.0,1.0,0.257797,0.0,0.0,1.0
12236,12236,15642099.0,0.658711,1.0,0.053571,0.3,0.000000,2.0,1.0,1.0,0.742837,1.0,0.0,0.0


1. Навчіть на цих даних класифікатор kNN з параметрами за замовченням і виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах. Зробіть заключення про отриману модель: вона хороша/погана, чи є high bias/high variance?

In [4]:
model_raw = KNeighborsClassifier()
model_raw.fit(data['train_x'], data['train_y'])

KNeighborsClassifier()

In [5]:
train_pred_proba = model_raw.predict_proba(data['train_x'])[:, 1]
val_pred_proba = model_raw.predict_proba(data['val_x'])[:, 1]

In [6]:
train_auc = roc_auc_score(data['train_y'], train_pred_proba)
val_auc = roc_auc_score(data['val_y'], val_pred_proba)

train_auc, val_auc

(np.float64(0.8033923325825397), np.float64(0.4973468687838672))

#Висновок до завдання 1
Модель показує непоганий результат (але і хорошим назвати цей результат тяжкувато) на тренувальних даних і зовсім поганий на валідаційних.

Можна було б сказати що модель перенавчена але результат на train не настільки хороший.

2. Використовуючи `GridSearchCV` знайдіть оптимальне значення параметра `n_neighbors` для класифікатора `kNN`. Псотавте крос валідацію на 5 фолдів.

  Після успішного завершення пошуку оптимального гіперпараметра
    - виведіть найкраще значення параметра
    - збережіть в окрему змінну `knn_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `knn_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пукнтом (2) цього завдання? Чи є вона краще за дерево прийняття рішень з попереднього ДЗ?

In [7]:
?GridSearchCV

In [8]:
search = GridSearchCV(
    KNeighborsClassifier(),
    {'n_neighbors': np.arange(1, 25)},
    cv=5,
    scoring="roc_auc",
)
search.fit(data['train_x'], data['train_y'])

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24])},
             scoring='roc_auc')

In [9]:
knn_best = search.best_estimator_

print(search.best_params_, search.best_score_)

{'n_neighbors': np.int64(16)} 0.5291322250727505


In [10]:
train_pred_proba_knn = knn_best.predict_proba(data['train_x'])[:, 1]
val_pred_proba_knn = knn_best.predict_proba(data['val_x'])[:, 1]

train_auc_knn = roc_auc_score(data['train_y'], train_pred_proba_knn)
val_auc_knn = roc_auc_score(data['val_y'], val_pred_proba_knn)

train_auc_knn, val_auc_knn

(np.float64(0.6868422094410794), np.float64(0.4909753755401605))

#Висновок до завдання 2

Модель стала гіршою -  значення в train суттєво погіршились. Результат значно гірший від результату по моделі з деревами.

3. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `GridSearchCV` за сіткою параметрів
  - `max_depth` від 1 до 20 з кроком 2
  - `max_leaf_nodes` від 2 до 10 з кроком 1

  Обовʼязково при цьому ініціюйте модель з фіксацією `random_state`.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_best` найкращу модель, знайдену з `GridSearchCV`
    - оцініть якість передбачень  `dt_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли вручну?

In [11]:
?GridSearchCV

In [12]:
model_dtc = GridSearchCV(
    DecisionTreeClassifier(),
    cv = 3,
    scoring = 'roc_auc',
    param_grid =
     {
        'max_depth': np.arange(1, 20, 2),
        'max_leaf_nodes': np.arange(2, 10, 1),
        'random_state': [42]
    }

)

In [13]:
%%time
model_dtc.fit(data['train_x'], data['train_y'])

CPU times: user 7.36 s, sys: 7.09 ms, total: 7.37 s
Wall time: 7.43 s


GridSearchCV(cv=3, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': array([ 1,  3,  5,  7,  9, 11, 13, 15, 17, 19]),
                         'max_leaf_nodes': array([2, 3, 4, 5, 6, 7, 8, 9]),
                         'random_state': [42]},
             scoring='roc_auc')

In [14]:
dt_best = model_dtc.best_estimator_
dt_train_proba = dt_best.predict_proba(data['train_x'])[:, 1]
dt_val_proba = dt_best.predict_proba(data['val_x'])[:, 1]

dt_train_auc = roc_auc_score(data['train_y'], dt_train_proba)
dt_val_auc = roc_auc_score(data['val_y'], dt_val_proba)

dt_train_auc, dt_val_auc

(np.float64(0.9001138615074584), np.float64(0.898395980519926))

#Висновок до завдання 3

Модель дерева прийняття рішень з пошуком за сіткою показує набагато кращий показник точності ніж попередні експерименти. І результат хороший як на тренувальному так і валідаційному наборах.

4. Виконайте пошук оптимальних гіперпараметрів для `DecisionTreeClassifier` з `RandomizedSearchCV` за заданою сіткою параметрів і кількість ітерацій 40.

  Поставте кросвалідацію на 3 фолди, `scoring='roc_auc'`, зафіксуйте `random_seed` процедури крос валідації та виміряйте, скільки часу потребує пошук оптимальних гіперпараметрів.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення параметра
    - збережіть в окрему змінну `dt_random_search_best` найкращу модель, знайдену з `RandomizedSearchCV`
    - оцініть якість передбачень  `dt_random_search_best` на тренувальній і валідаційній вибірці з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи ця модель краща за ту, що ви знайшли з `GridSearch`?
    - проаналізуйте параметри `dt_random_search_best` і порівняйте з параметрами `dt_best` - яку бачите відмінність? Ця вправа потрібна аби зрозуміти, як різні налаштування `DecisionTreeClassifier` впливають на якість моделі.

In [15]:
params_dt = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': np.arange(1, 20),
    'max_leaf_nodes': np.arange(2, 20),
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

In [16]:
model_dt_r = RandomizedSearchCV(
    DecisionTreeClassifier(),
    param_distributions = params_dt,
    scoring = 'roc_auc',
    n_iter = 40,
    random_state = 42
)

In [17]:
%%time
model_dt_r.fit(data['train_x'], data['train_y'])

CPU times: user 4.1 s, sys: 5.43 ms, total: 4.11 s
Wall time: 4.25 s


RandomizedSearchCV(estimator=DecisionTreeClassifier(), n_iter=40,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19]),
                                        'max_features': [None, 'sqrt', 'log2'],
                                        'max_leaf_nodes': array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19]),
                                        'min_samples_leaf': [1, 2, 4, 8],
                                        'min_samples_split': [2, 5, 10, 20],
                                        'splitter': ['best', 'random']},
                   random_state=42, scoring='roc_auc')

In [18]:
dt_random_search_best = model_dt_r.best_estimator_
dt_r_train_proba = dt_random_search_best.predict_proba(data['train_x'])[:, 1]
dt_r_val_proba = dt_random_search_best.predict_proba(data['val_x'])[:, 1]

dt_r_train_auc = roc_auc_score(data['train_y'], dt_r_train_proba)
dt_r_val_auc = roc_auc_score(data['val_y'], dt_r_val_proba)

dt_r_train_auc, dt_r_val_auc

(np.float64(0.9169275635848141), np.float64(0.9166204815145071))

In [19]:
model_dt_r.best_params_, model_dtc.best_params_

({'splitter': 'best',
  'min_samples_split': 20,
  'min_samples_leaf': 2,
  'max_leaf_nodes': np.int64(14),
  'max_features': None,
  'max_depth': np.int64(16),
  'criterion': 'entropy'},
 {'max_depth': np.int64(5), 'max_leaf_nodes': np.int64(9), 'random_state': 42})

Висновок до завдання 4

Модель RandomSearchCV показала ще кращий результат, особбливо на валідаційних даних (хоч тільки 2 пп, але на такій точності це вже має вагоміше значення).
Видно що рандом сьорч відрізняється в максимальній глибині (16 проти 5) і в максимальній глибині нодів (14 проти 9).

5. Якщо у Вас вийшла метрика `AUROC` в цій серії експериментів - зробіть ще один `submission` на Kaggle і додайте код для цього і скріншот скора на публічному лідерборді нижче.

  Сподіваюсь на цьому етапі ви вже відчули себе справжнім дослідником 😉

In [20]:
test_df = pd.read_csv('drive/MyDrive/ML for people/test.csv')
test_df.head()

,id,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,15000,15594796.0,Chu,584.0,Germany,Male,30.0,2.0,146053.66,1.0,1.0,1.0,157891.86
1,15001,15642821.0,Mazzi,551.0,France,Male,39.0,5.0,0.00,2.0,1.0,1.0,67431.28
2,15002,15716284.0,Onyekachi,706.0,France,Male,43.0,8.0,0.00,2.0,1.0,0.0,156768.45
3,15003,15785078.0,Martin,717.0,Spain,Male,45.0,3.0,0.00,1.0,1.0,1.0,166909.87
4,15004,15662955.0,Kenechukwu,592.0,Spain,Male,43.0,8.0,0.00,2.0,1.0,1.0,143681.97


In [21]:
test_df_prep = pp.preprocess_new_data(test_df, scaler=data['scaler'], encoder=data['encoder'])

/content/process_bank_churn.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_new['Gender'] = df_new['Gender'].replace({'Male': 1, 'Female': 0})


In [22]:
dt_random_search_best.fit(data['train_x'], data['train_y'])

DecisionTreeClassifier(criterion='entropy', max_depth=np.int64(16),
                       max_leaf_nodes=np.int64(14), min_samples_leaf=2,
                       min_samples_split=20)

In [24]:
sample_sub = pd.read_csv('drive/MyDrive/ML for people/sample_submission.csv')

y_proba = dt_random_search_best.predict_proba(test_df_prep)[:, 1]
test_df_prep['Exited'] = y_proba

In [25]:
sample_sub = sample_sub.merge(test_df_prep[["id", "Exited"]], on="id", how="left", suffixes=("", "_new"))
sample_sub["Exited"] = sample_sub["Exited_new"]
sample_sub = sample_sub.drop(columns="Exited_new")

sample_sub.head()

,id,Exited
0,15000,0.237911
1,15001,0.012115
2,15002,0.203947
3,15003,0.569848
4,15004,0.082171


In [26]:
sample_sub.to_csv("submission_log_reg.csv", index=False, encoding="utf-8")